<a href="https://colab.research.google.com/github/Haross/sql_notebook/blob/main/SQL_Practice_All_topics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Practice - All topics


## **SQL Environment Setup (do not edit)**

In [1]:
# @title

%%capture
!mkdir -p notebook_lib
!wget --cache=off -q -O notebook_lib/sql_runner.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner.py
!wget --cache=off -q -O notebook_lib/sql_runner_store.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_store.py
!wget --cache=off -q -O notebook_lib/sql_runner_ui_bits.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_ui_bits.py
!wget --cache=off -q -O notebook_lib/validators.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/validators.py

!wget --cache=off -q -O notebook_lib/cloud_submitter.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/cloud_submitter.py

!touch notebook_lib/__init__.py
#!pip install -q duckdb

from notebook_lib.sql_runner import make_sql_runner
from notebook_lib.validators import make_df_validator_nospoilers, check_process_rules
from notebook_lib.cloud_submitter import make_cloud_run_submitter
import duckdb
import pandas as pd
from pathlib import Path


In [2]:
# @title

DB_FILE = 'practice_sql.duckdb'

if DB_FILE != ":memory:":
    Path(DB_FILE).unlink(missing_ok=True)

conn = duckdb.connect(DB_FILE)

conn.execute(r'''
DROP TABLE IF EXISTS clothing_order;
DROP TABLE IF EXISTS clothing;
DROP TABLE IF EXISTS customer;
DROP TABLE IF EXISTS color;
DROP TABLE IF EXISTS category;

CREATE TABLE category (
    id INTEGER PRIMARY KEY,
    name VARCHAR,
    parent_id INTEGER NULL
);

CREATE TABLE color (
    id INTEGER PRIMARY KEY,
    name VARCHAR,
    extra_fee INTEGER
);

CREATE TABLE customer (
    id INTEGER PRIMARY KEY,
    first_name VARCHAR,
    last_name VARCHAR,
    favorite_color_id INTEGER
);

CREATE TABLE clothing (
    id INTEGER PRIMARY KEY,
    name VARCHAR,
    size VARCHAR,
    price DECIMAL(10,2),
    color_id INTEGER,
    category_id INTEGER
);

CREATE TABLE clothing_order (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    clothing_id INTEGER,
    items INTEGER,
    order_date DATE
);

INSERT INTO category (id, name, parent_id) VALUES
(1, 'sportswear', NULL),
(2, 'tracksuits', 1),
(3, 'football', 1),
(4, 'swimsuits', 1),
(5, 'ski suits', 1),
(6, 'casual fashion', NULL),
(7, 'women fashion', 6),
(8, 'men fashion', 6),
(9, 'women', 2),
(10, 'men', 2),
(11, 'women', 3),
(12, 'men', 3),
(13, 'women', 4),
(14, 'men', 4),
(15, 'women', 5),
(16, 'men', 5),
(17, 'athletic', 1),
(18, 'women', 17),
(19, 'men', 17),
(20, 'fitness fashion', 1),
(21, 'women', 20),
(22, 'men', 20),
(23, 'T-shirt', 7),
(24, 'T-shirt', 8),
(25, 'short', 7),
(26, 'short', 8),
(27, 'leggings', 7),
(28, 'trousers', 8),
(29, 'leggings', 21),
(30, 'trousers', 21),
(31, 'trousers', 22),
(32, 'tops', 21),
(33, 'trousers', 17),
(34, 'trousers', 17);

INSERT INTO color (id, name, extra_fee) VALUES
(1, 'white', 0),
(2, 'black', 1),
(3, 'navy', 5),
(4, 'blue', 7),
(5, 'light blue', 7),
(6, 'pink', 10),
(7, 'fuchsia', 12),
(8, 'green', 5),
(9, 'dark green', 5),
(10, 'light green', 12),
(11, 'violet', 4),
(12, 'orange', 9),
(13, 'yellow', 9),
(14, 'brown', 11),
(15, 'olive', 15),
(16, 'gray', 2),
(17, 'red', 16),
(18, 'lavender', 17),
(19, 'teal', 13);

INSERT INTO customer (id, first_name, last_name, favorite_color_id) VALUES
(1, 'Liam', 'Johnson', 19),
(2, 'Noah', 'Johnson', 4),
(3, 'Oliver', 'Williams', 12),
(4, 'William', 'Brown', 11),
(5, 'Elijah', 'Jones', 19),
(6, 'James', 'Garcia', 16),
(7, 'Benjamin', 'Miller', 4),
(8, 'Lucas', 'Davis', 12),
(9, 'Michael', 'Wilson', 18),
(10, 'Daniel', 'Anderson', 6),
(11, 'Logan', 'Thomas', 15),
(12, 'Jackson', 'Taylor', 9),
(13, 'Sebastian', 'Moore', 14),
(14, 'Jack', 'Jackson', 8),
(15, 'Aiden', 'Martin', 6),
(16, 'Owen', 'Lee', 1),
(17, 'Samuel', 'Perez', 17),
(18, 'Matthew', 'Thompson', 19),
(19, 'Joseph', 'White', 18),
(20, 'Levi', 'Harris', 12),
(21, 'Carter', 'Robinson', 13),
(22, 'Julian', 'Walker', 16),
(23, 'Luke', 'Young', 10),
(24, 'Grayson', 'Allen', 4),
(25, 'Isaac', 'King', 5),
(26, 'Jayden', 'Wright', 3),
(27, 'Theodore', 'Scott', 1),
(28, 'Gabriel', 'Torres', 8),
(29, 'Sophia', 'Hall', 5),
(30, 'Charlotte', 'Rivera', 17),
(31, 'Mia', 'Campbell', 2),
(32, 'Amelia', 'Mitchell', 17),
(33, 'Harper', 'Carter', 6),
(34, 'Evelyn', 'Allen', 3),
(35, 'Abigail', 'King', 19),
(36, 'Emily', 'Wright', 5),
(37, 'Elizabeth', 'Green', 8),
(38, 'Mila', 'Jones', 3),
(39, 'Ella', 'Jones', 15),
(40, 'Scarlett', 'Walker', 16),
(41, 'Victoria', 'Young', 7),
(42, 'Madison', 'Young', 6),
(43, 'Luna', 'Young', 1),
(44, 'Lily', 'Jackson', 18),
(45, 'Eleanor', 'Ramirez', 19),
(46, 'Hannah', 'Jackson', 16),
(47, 'Lillian', 'Thompson', 19),
(48, 'Addison', 'Brown', 17),
(49, 'Aubrey', 'Jones', 14),
(50, 'Ellie', 'Garcia', 12),
(51, 'Stella', 'Nguyen', 19),
(52, 'Natalie', 'Williams', 18),
(53, 'Zoe', 'Smith', 16),
(54, 'Leah', 'Green', 1),
(55, 'Hazel', 'Smith', 17),
(56, 'Violet', 'Williams', 10),
(57, 'Aurora', 'Williams', 9),
(58, 'Savannah', 'Smith', 17),
(59, 'Audrey', 'Nguyen', 12),
(60, 'Brooklyn', 'Williams', 19),
(61, 'Bella', 'Brown', 13),
(62, 'Claire', 'Jones', 10),
(63, 'Skylar', 'Garcia', 6),
(64, 'Lucy', 'Jackson', 10),
(65, 'Paisley', 'Brown', 8),
(66, 'Everly', 'Garcia', 8),
(67, 'Kennedy', 'Nguyen', 3),
(68, 'Samantha', 'Harris', 10);

INSERT INTO clothing (id, name, size, price, color_id, category_id) VALUES
(1, 'band 22d Sport', 'S', 12, 6, 10),
(2, 'leggings X22', 'M', 23, 13, 27),
(3, 'membrane jacket C345', 'L', 154, 6, 2),
(4, 'sport membrane trousers', 'XL', 160, 19, 33),
(5, 'jacket 244WD', 'XXl', 450, 18, 5),
(6, 'sweatshirt ski sport men', '3XL', 125, 12, 6),
(7, 'new fashion trouesrs DF', 'S', 245, 19, 34),
(8, 'modern fashion trouesrs XV', 'M', 320, 11, 9),
(9, 'short trusers', 'L', 167, 13, 28),
(10, 'shirt blue sky 2', 'XL', 89, 2, 7),
(11, 'leggings summer', 'XXl', 122, 3, 27),
(12, 'winter jacket CX', '3XL', 570, 7, 10),
(13, 'cotton T-shirt sunrise', 'S', 54, 13, 24),
(14, 'ski jacket 3-layeres', 'M', 760, 6, 16),
(15, 'trousers woman 30', 'L', 160, 17, 30),
(16, 'jeans W200', 'XL', 80, 9, 28),
(17, 'leggings jeans', 'XXl', 125, 3, 27),
(18, 'high-visibility vest', '3XL', 350, 4, 10),
(19, 'finess leggings MEN', 'S', 124, 10, 31),
(20, 'wind jacket', 'M', 455, 4, 16),
(21, 'sport bra', 'L', 90, 13, 20),
(22, 'Eco T-shirt', 'XL', 134, 15, 24),
(23, 'layered swim skirt', 'XXl', 800, 3, 4),
(24, 'sweatshirt eco style', '3XL', 230, 6, 22),
(25, 'joggers pants', 'S', 110, 2, 31),
(26, 'Eco T-shirt cotton', 'M', 24, 5, 24),
(27, 'short summer 2021', 'L', 53, 7, 4),
(28, 'layered leggings M2021', 'XL', 225, 7, 31),
(29, 'layered leggings M2022', 'XXl', 440, 19, 31),
(30, 'breathable longsleeve blouse', '3XL', 245, 11, 18),
(31, 'breathable longsleeve blouse', 'S', 268, 14, 14),
(32, 'cotton short', 'M', 178, 14, 25),
(33, 'joggers cotton', 'L', 228, 6, 31),
(34, 'linen T-shirt', 'XL', 59, 14, 23),
(35, 'lightwear jacket', 'XXl', 780, 9, 5),
(36, 'linen pants', '3XL', 550, 8, 8),
(37, 'breathable T-shirt', 'S', 68, 19, 23),
(38, 'breathable jacket', 'M', 670, 19, 8),
(39, 'layered skirt 2021', 'L', 320, 18, 13),
(40, 'joggers summer 2021', 'XL', 199, 18, 21),
(41, 'breathable T-shirt', 'XXl', 45, 7, 24),
(42, 'hiking jacket', '3XL', 450, 1, 10),
(43, 'lightwear breathable blouse', 'S', 455, 11, 15),
(44, 'lightwear breathable blouse', 'M', 355, 10, 16),
(45, 'cotton socks', 'L', 15, 7, 3),
(46, 'leggings summer2022', 'XL', 340, 16, 31),
(47, 'pants slim', 'XXl', 225, 4, 3),
(48, 'pants cotton', '3XL', 260, 14, 3),
(49, 'swimming trunks', 'S', 432, 19, 14),
(50, 'socks breathable cotton', 'M', 130, 2, 12),
(51, 'cotton run cap', 'L', 75, 4, 17),
(52, 'shorts 2021', 'XL', 12, 4, 11),
(53, 'leggings winter 22', 'XXl', 34, 5, 27),
(54, 'shorts 2021 men', '3XL', 210, 16, 12),
(55, 'band 2021', 'S', 17, 4, 18),
(56, 'cotton top', 'M', 186, 16, 32),
(57, 'summer jacket 250WD', 'L', 657, 15, 9),
(58, 'layered trousers winter', 'XL', 340, 6, 28),
(59, 'sockscotton sport', 'XXl', 34, 8, 12),
(60, 'jacket 2000', '3XL', 870, 6, 8),
(61, 'swim shorts summer', 'S', 145, 17, 14),
(62, 'shirt blue sky 2', 'M', 225, 16, 8),
(63, 'men joggers cotton', 'L', 165, 9, 25),
(64, 'women joggers cotton', 'XL', 238, 16, 7),
(65, 'layered leggings', 'XXl', 554, 8, 33),
(66, 'jacket 2000', '3XL', 799, 2, 2),
(67, 'jacket summer 2021', 'S', 699, 16, 2),
(68, 'shorts 2000', 'M', 43, 11, 25),
(69, 'T-shirt Sun', 'L', 55, 6, 23),
(70, 'layered cotton trousers', 'XL', 270, 16, 33),
(71, 'top summer 2021', 'XXl', 126, 19, 13),
(72, 'socks 2000', '3XL', 19, 13, 12),
(73, 'evening dress', 'S', 300, 17, 7),
(74, 'jacket DNW2', 'M', 400, 16, 2),
(75, 'leggings quarter', 'L', 100, 5, 29),
(76, 'trousers 2_3', 'XL', 299, 6, 33),
(77, 'short summer 2022', 'XXl', 99, 14, 14),
(78, 'modern 2022 shorts', '3XL', 87, 1, 12),
(79, 'cotton universal socks', 'S', 67, 13, 17),
(80, 'T-shirt modern summer', 'M', 42, 7, 24),
(81, 'socks soprt 2000', 'L', 32, 4, 17),
(82, 'short linen', 'XL', 179, 3, 26),
(83, 'short layered', 'XXl', 77, 3, 19),
(84, 'socks women', '3XL', 65, 2, 21),
(85, 'socks women light', 'S', 49, 5, 18),
(86, 'eco joggers', 'M', 189, 16, 21),
(87, 'eco cotton trousers DXS1', 'L', 279, 1, 34),
(88, 'eco athletic jacket', 'XL', 769, 19, 18),
(89, 'jacket winter 2021', 'XXl', 1200, 7, 16),
(90, 'socks modern men', '3XL', 78, 6, 22),
(91, 'jacket winter 2022', 'S', 450, 13, 5),
(92, 'cotton sport band', 'M', 25, 8, 18),
(93, 'shirt modern 2021', 'L', 39, 11, 7),
(94, 'light sport trousers', 'XL', 155, 18, 30),
(95, 'cotton sport socks', 'XXl', 120, 14, 3),
(96, 'socks 2021', '3XL', 87, 13, 19),
(97, 'socks 2022', 'S', 75, 5, 3),
(98, 'socks 2021 F', 'M', 64, 18, 11),
(99, 'socks 2022 FC', 'L', 110, 9, 7),
(100, 'sweatshirt ski sport women', 'S', 175, 10, 6);

INSERT INTO clothing_order (id, customer_id, clothing_id, items, order_date) VALUES
(1, 2, 98, 3, '2021-05-10'),
(2, 11, 86, 4, '2021-05-25'),
(3, 48, 16, 3, '2021-06-16'),
(4, 40, 56, 3, '2021-06-03'),
(5, 42, 3, 1, '2021-05-22'),
(6, 18, 73, 4, '2021-06-17'),
(7, 56, 9, 2, '2021-06-10'),
(8, 33, 97, 4, '2021-05-08'),
(9, 47, 97, 5, '2021-06-13'),
(10, 64, 52, 4, '2021-06-17'),
(11, 45, 88, 3, '2021-05-25'),
(12, 9, 76, 1, '2021-05-12'),
(13, 30, 13, 3, '2021-05-11'),
(14, 12, 56, 2, '2021-05-26'),
(15, 7, 13, 4, '2021-06-10'),
(16, 58, 87, 2, '2021-05-05'),
(17, 11, 59, 2, '2021-05-19'),
(18, 35, 32, 3, '2021-06-27'),
(19, 21, 94, 5, '2021-06-01'),
(20, 52, 76, 3, '2021-06-10'),
(21, 44, 59, 2, '2021-05-12'),
(22, 6, 79, 5, '2021-05-16'),
(23, 28, 49, 4, '2021-06-11'),
(24, 56, 10, 4, '2021-05-23'),
(25, 54, 48, 5, '2021-05-14'),
(26, 35, 55, 5, '2021-05-19'),
(27, 7, 98, 4, '2021-05-08'),
(28, 14, 87, 3, '2021-05-09'),
(29, 29, 3, 2, '2021-06-19'),
(30, 7, 41, 5, '2021-05-11'),
(31, 27, 57, 3, '2021-05-07'),
(32, 58, 31, 2, '2021-05-25'),
(33, 28, 42, 2, '2021-06-04'),
(34, 16, 7, 2, '2021-06-08'),
(35, 61, 2, 2, '2021-06-21'),
(36, 34, 66, 5, '2021-06-18'),
(37, 4, 53, 3, '2021-05-07'),
(38, 16, 69, 2, '2021-05-20'),
(39, 48, 51, 4, '2021-05-15'),
(40, 17, 30, 5, '2021-05-24'),
(41, 37, 40, 5, '2021-06-04'),
(42, 12, 36, 2, '2021-05-30'),
(43, 1, 44, 4, '2021-05-14'),
(44, 30, 77, 1, '2021-05-10'),
(45, 38, 65, 5, '2021-06-02'),
(46, 54, 87, 2, '2021-06-14'),
(47, 29, 66, 1, '2021-05-08'),
(48, 54, 62, 5, '2021-05-13'),
(49, 59, 28, 1, '2021-05-31'),
(50, 68, 32, 4, '2021-06-01'),
(51, 68, 47, 2, '2021-06-04'),
(52, 21, 43, 5, '2021-06-13'),
(53, 12, 48, 1, '2021-05-02'),
(54, 17, 26, 3, '2021-05-25'),
(55, 53, 23, 5, '2021-06-28'),
(56, 30, 61, 1, '2021-05-09'),
(57, 47, 24, 1, '2021-06-25'),
(58, 56, 93, 5, '2021-06-15'),
(59, 28, 52, 1, '2021-06-12'),
(60, 52, 97, 1, '2021-05-14'),
(61, 4, 56, 3, '2021-06-22'),
(62, 65, 43, 2, '2021-06-07'),
(63, 66, 25, 1, '2021-06-10'),
(64, 40, 85, 3, '2021-05-17'),
(65, 26, 69, 2, '2021-05-18'),
(66, 21, 17, 2, '2021-05-29'),
(67, 9, 45, 1, '2021-06-21'),
(68, 4, 68, 5, '2021-06-05'),
(69, 24, 99, 2, '2021-05-22'),
(70, 7, 17, 2, '2021-06-08'),
(71, 48, 23, 5, '2021-05-07'),
(72, 58, 32, 5, '2021-05-29'),
(73, 38, 96, 5, '2021-05-20'),
(74, 24, 97, 5, '2021-06-04'),
(75, 64, 94, 5, '2021-06-05'),
(76, 2, 13, 1, '2021-05-11'),
(77, 51, 41, 2, '2021-06-21'),
(78, 18, 85, 2, '2021-05-04'),
(79, 43, 33, 4, '2021-05-02'),
(80, 57, 88, 3, '2021-06-24'),
(81, 38, 25, 1, '2021-06-28'),
(82, 30, 4, 3, '2021-05-15'),
(83, 25, 93, 3, '2021-05-24'),
(84, 57, 74, 5, '2021-06-15'),
(85, 62, 92, 5, '2021-05-10'),
(86, 55, 81, 2, '2021-05-23'),
(87, 21, 41, 4, '2021-05-07'),
(88, 21, 11, 5, '2021-06-16'),
(89, 6, 56, 1, '2021-06-17'),
(90, 18, 29, 1, '2021-06-23'),
(91, 9, 16, 5, '2021-05-06'),
(92, 59, 21, 2, '2021-06-08'),
(93, 24, 52, 4, '2021-05-08'),
(94, 22, 29, 5, '2021-05-22'),
(95, 48, 41, 2, '2021-05-19'),
(96, 22, 84, 2, '2021-06-28'),
(97, 64, 16, 5, '2021-05-14'),
(98, 47, 51, 2, '2021-05-10'),
(99, 48, 49, 2, '2021-05-18'),
(100, 64, 95, 5, '2021-05-09');

-- =========================================================
-- EXTRA DATA FOR WHERE EXERCISES
-- =========================================================

-- -------------------------
-- category
-- Helps exercise_4 with clearer pass/fail examples
-- -------------------------
INSERT INTO category (id, name, parent_id) VALUES
(35, 'women', 6),      -- passes Part A, fails Part B
(36, 'women', 1),      -- passes Part A, fails Part B
(37, 'trousers', 1),   -- passes Part A, fails Part C
(38, 'tops', 1),       -- passes Part A
(39, 'trousers', 17),  -- passes Part C only
(40, 'men casual', 6); -- passes Part A


-- -------------------------
-- color
-- Helps exercise_3 with explicit pass/fail rows
-- -------------------------
INSERT INTO color (id, name, extra_fee) VALUES
(20, 'green', 7),    -- fails Part B
(21, 'pink', 9),     -- fails Part C
(22, 'pink', 10),    -- passes Part C
(23, 'white', 5),    -- passes Part A
(24, 'black', 14);   -- fails Part A


-- -------------------------
-- customer
-- Helps exercise_2 and exercise_6_3
-- -------------------------
INSERT INTO customer (id, first_name, last_name, favorite_color_id) VALUES
(69, 'Charlotte', 'Brown', 17),   -- ex2: passes Part A + Part B
(70, 'Emily', 'Brown', 17),       -- ex2: passes Part A, fails Part B
(71, 'Liam', 'Jones', 19),        -- ex2: passes Part C only
(72, 'Luna', 'Smith', 19),        -- ex6_3: passes Part B only
(73, 'Avery', 'Jackson', 18),     -- ex6_3: passes Part A + Part B
(74, 'Lily', 'Jones', 19),        -- ex6_3: fails Part B
(75, 'Ella', 'Jones', 17),        -- ex6_3: passes Part C
(76, 'Amelia', 'Brown', 17),      -- ex6_3: fails Part C
(77, 'Emily', 'Brown', 17),       -- ex6_3: fails Part C because exact Emily
(78, 'Mia', 'Brown', 4),          -- ex6_3: fails Part A (name too short)
(79, 'Lucas', 'Taylor', 4),       -- ex6_3: fails Part A (last name not allowed)
(80, 'Liam', 'Johnson', 19);      -- ex6_3: passes Part A + Part B


-- -------------------------
-- clothing
-- Helps exercise_1, exercise_6_1, exercise_6_2
-- -------------------------
INSERT INTO clothing (id, name, size, price, color_id, category_id) VALUES
(101, 'navy tee 130', 'M', 130, 3, 24),          -- ex1: passes Part C
(102, 'navy tee 100', 'L', 100, 3, 24),          -- ex1: fails Part C
(103, 'pink blouse 220', 'M', 220, 6, 7),        -- ex1: passes Part B
(104, 'pink blouse 150', 'L', 150, 6, 7),        -- ex1: fails Part B
(105, 'basic top small', 'S', 120, 4, 7),        -- ex1: fails Part A
(106, 'basic top large', 'L', 120, 4, 7),        -- ex1: passes Part A

(107, 'cotton shirt black', 'M', 80, 2, 7),      -- ex6_1: passes Part B
(108, 'modern shirt', 'M', 80, 2, 7),            -- ex6_1: fails Part B
(109, 'wind jacket', 'M', 650, 4, 8),            -- ex6_1: passes Part C
(110, 'winter jacket', 'L', 720, 4, 8),          -- ex6_1: fails Part C
(111, 'eco jacket pro', 'M', 500, 4, 22),        -- ex6_1: passes Part C
(112, 'travel jacket', 'M', 500, 4, 21),         -- ex6_1: fails Part C

(113, 'cotton T-shirt lab', 'M', 55, 13, 23),    -- ex6_2: passes Part B
(114, 'shirt white classic', 'M', 60, 1, 24),    -- ex6_2: passes Part B
(115, 'shirt black classic', 'M', 60, 2, 24),    -- ex6_2: fails Part B
(116, 'winter shirt white', 'M', 60, 1, 24),     -- ex6_2: fails Part B
(117, 'red swim pro', 'M', 350, 17, 24),         -- ex6_2: passes Part C
(118, 'pink elite top', 'M', 455, 6, 23),        -- ex6_2: passes Part C
(119, 'fuchsia mid top', 'M', 200, 7, 23),       -- ex6_2: fails Part C
(120, 'red summer wave', 'M', 350, 17, 24);      -- ex6_2: fails Part C


-- -------------------------
-- clothing_order
-- Helps exercise_5
-- -------------------------
INSERT INTO clothing_order (id, customer_id, clothing_id, items, order_date) VALUES
(101, 10, 97, 5, '2021-05-20'),   -- passes Part B
(102, 11, 50, 5, '2021-05-20'),   -- fails Part B
(103, 48, 23, 3, '2021-06-05'),   -- passes Part C
(104, 48, 50, 3, '2021-06-05'),   -- fails Part C
(105, 48, 50, 2, '2021-06-06'),   -- passes Part C
(106, 12, 30, 2, '2021-05-09'),   -- fails Part A
(107, 13, 31, 2, '2021-05-10'),   -- passes Part A
(108, 14, 32, 2, '2021-06-10'),   -- passes Part A
(109, 15, 33, 2, '2021-06-11');   -- fails Part A
''')
print(f"Duckdb database ready ✅ ({DB_FILE})")


Duckdb database ready ✅ (practice_sql.duckdb)


## Sportswear Database — Tables Overview

The sportswear database stores information about products, customers, and orders.  
It consists of five tables:

- `color`
- `customer`
- `category`
- `clothing`
- `clothing_order`

Let’s take a closer look at each of them.

---

### `color`

The table `color` contains information about the available colors for sportswear items.

It has the following columns:

- `id` – the unique identifier of each color  
- `name` – the name of the color  
- `extra_fee` – any additional cost for choosing this color  

---

### `customer`

The table `customer` stores information about customers.

It contains the following columns:

- `id` – the unique identifier of the customer  
- `first_name` – the customer’s first name  
- `last_name` – the customer’s last name  
- `favorite_color_id` – the customer’s favorite color (references `color.id`)  

---

### `category`

The table `category` defines product categories and subcategories.

It contains:

- `id` – the unique identifier of the category  
- `name` – the name of the category  
- `parent_id` – the id of the parent category  

If `parent_id` is `NULL`, the category is a **main category**. Otherwise, it is a **subcategory** of another category in the same table.

---

### `clothing`

The table `clothing` stores information about the products available in the shop.

It contains:

- `id` – the unique identifier of the item  
- `name` – the name of the item  
- `size` – the size of the item (S, M, L, XL, XXL, 3XXL)  
- `price` – the price of the item  
- `color_id` – the color of the item (references `color.id`)  
- `category_id` – the category of the item (references `category.id`)  

---

### `clothing_order`

Finally, the table `clothing_order` stores information about customer orders.

It contains:

- `id` – the unique identifier of the order  
- `customer_id` – the customer placing the order (references `customer.id`)  
- `clothing_id` – the ordered item (references `clothing.id`)  
- `items` – the number of items ordered  
- `order_date` – the date of the order  


# Conditions practice

### Note about the examples in exercises 1 to 6

The examples shown in each exercise are included only to clarify the logic of each individual part.

- When an example says **“should pass this part”** or **“should fail this part”**, it refers only to that one part of the `WHERE` condition.
- Since all parts must be joined with `AND`, an example may pass one part but still not appear in the final result if it fails another part.
- Each exercise uses only one table, so the examples are written using attributes from that same table only.
- For clarity, some examples use meaningful values such as color names like `pink`, `black`, or `navy`, even when the actual table stores those values as IDs such as `color_id`. This is only to make the logic easier to read.


In [33]:
# @title ER diagram
%%html
<img id="er-img" style="width:70%; max-width:100%;  height:auto;"
     data-light="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/sql_final_practice_black.png"
     data-dark="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/sql_final_practice_black.png"
     alt="ER diagram">

<script>
  const img = document.getElementById("er-img");

  function isDarkTheme() {
    // Colab sets html[theme=dark] on the top document
    const themeAttr = document.documentElement.getAttribute("theme");
    if (themeAttr) return themeAttr === "dark";

    // fallback: OS/browser preference
    return window.matchMedia && window.matchMedia("(prefers-color-scheme: dark)").matches;
  }

  function updateImage() {
    img.src = isDarkTheme() ? img.dataset.dark : img.dataset.light;
  }

  updateImage();

  // React to Colab theme toggles (attribute changes)
  new MutationObserver(updateImage).observe(document.documentElement, {
    attributes: true,
    attributeFilter: ["theme"]
  });

  // React to OS/browser theme changes (fallback)
  if (window.matchMedia) {
    const mq = window.matchMedia("(prefers-color-scheme: dark)");
    mq.addEventListener?.("change", updateImage);
    mq.addListener?.(updateImage); // older browsers
  }
</script>

In [4]:
# @title Exercise 1
base_exercise_1 = make_df_validator_nospoilers(
    expected_hash='4c83f8066cb73ee206ba489b1840472e60e43bad8a280cc2e5aada49d44964f7',
    required_cols=['id', 'name', 'size', 'price', 'color_id', 'category_id'],
    expected_rows=42,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_1 = base_exercise_1

make_sql_runner(
    conn,
    runner_id="exercise_1",
    description_md="### Exercise 1\n\n  Write a SQL query for table `clothing` that fulfills the following conditions.  \n  All condition parts must be joined using `AND`.\n\n  - Part A: select clothing items with size `M` or `L`\n    - Example that should pass this part: `size = 'M'`\n    - Example that should fail this part: `size = 'S'`\n\n  - Part B: exclude items with color `pink`, unless the price is at least `200`\n    - Example that should pass this part: `color = 'pink', price = 220`\n    - Example that should fail this part: `color = 'pink', price = 150`\n\n  - Part C: exclude items from category `24`, except when the color is `navy` and the price is either below `50` or above `120`\n    - Example that should pass this part: `category_id = 24, color = 'navy', price = 130`\n    - Example that should fail this part: `category_id = 24, color = 'navy', price = 100`\n\n  Return: `id`, `name`, `size`, `price`, `color_id`, `category_id`\n",
    validator=val_exercise_1,
    sol_sql="SELECT id, name, size, price, color_id, category_id FROM clothing WHERE size IN ('M', 'L') AND (color_id <> 6 OR price >= 200) AND ( category_id <> 24 OR ( category_id = 24 AND color_id = 3 AND (price < 50 OR price > 120) ) )",
    select_only=True,
    dedupe=True,
    schema_tables=['clothing']
)


In [5]:
# @title Exercise 2
base_exercise_2 = make_df_validator_nospoilers(
    expected_hash='d8a56f382383ba9ddf5bab06a409b2057e811c52ecdd68f08c00ed7a34265433',
    required_cols=['id', 'first_name', 'last_name', 'favorite_color_id'],
    expected_rows=10,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_2 = base_exercise_2

make_sql_runner(
    conn,
    runner_id="exercise_2",
    description_md="### Exercise 2\n\nWrite a SQL query for table `customer` that fulfills the following conditions.  \nAll condition parts must be joined using `AND`.\n\n- Part A: select customers whose last name is `Johnson`, `Williams`, or `Brown`\n  - Example that should pass this part: `last_name = 'Johnson'`\n  - Example that should fail this part: `last_name = 'Smith'`\n\n- Part B: exclude customers whose favorite color is `red`, unless the first name is `Charlotte`\n  - Example that should pass this part: `favorite color = 'red', first_name = 'Charlotte'`\n  - Example that should fail this part: `favorite color = 'red', first_name = 'Emily'`\n\n- Part C: exclude customers whose favorite color is `teal`, except when the last name is `Jones` and the first name is not `Elijah` and not `Aubrey`\n  - Example that should pass this part: `favorite color = 'teal', last_name = 'Jones', first_name = 'Liam'`\n  - Example that should fail this part: `favorite color = 'teal', last_name = 'Jones', first_name = 'Elijah'`\n\nReturn: `id`, `first_name`, `last_name`, `favorite_color_id`\n",
    validator=val_exercise_2,
    sol_sql="SELECT id, first_name, last_name, favorite_color_id FROM customer WHERE last_name IN ('Johnson', 'Williams', 'Brown') AND (favorite_color_id <> 17 OR first_name = 'Charlotte') AND ( favorite_color_id <> 19 OR ( favorite_color_id = 19 AND last_name = 'Jones' AND first_name NOT IN ('Elijah', 'Aubrey') ) )",
    select_only=True,
    dedupe=True,
    schema_tables=['customer']
)


In [6]:
# @title Exercise 3
base_exercise_3 = make_df_validator_nospoilers(
    expected_hash='4825dd8fc7cfcf85f7b79541a49f48bd3b976c8c52bf55bf1ea500a034ce1b38',
    required_cols=['id', 'name', 'extra_fee'],
    expected_rows=13,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_3 = base_exercise_3

make_sql_runner(
    conn,
    runner_id="exercise_3",
    description_md="description_md: |\n  ### Exercise 3\n\n  Write a SQL query for table `color` that fulfills the following conditions.  \n  All condition parts must be joined using `AND`.\n\n  - Part A: select colors with an extra fee from `5` through `13`\n    - Example that should pass this part: `extra_fee = 5`\n    - Example that should fail this part: `extra_fee = 4`\n\n  - Part B: exclude the color `green`, unless its extra fee is exactly `5`\n    - Example that should pass this part: `name = 'green', extra_fee = 5`\n    - Example that should fail this part: `name = 'green', extra_fee = 7`\n\n  - Part C: exclude `pink`, `fuchsia`, and `lavender`, except when the extra fee is at least `17`, or when the color is `pink` and the extra fee is exactly `10`\n    - Example that should pass this part: `name = 'pink', extra_fee = 10`\n    - Example that should fail this part: `name = 'fuchsia', extra_fee = 12`\n\n  Return: `id`, `name`, `extra_fee`\n",
    validator=val_exercise_3,
    sol_sql="SELECT id, name, extra_fee FROM color WHERE extra_fee BETWEEN 5 AND 13 AND (name <> 'green' OR extra_fee = 5) AND ( name NOT IN ('pink', 'fuchsia', 'lavender') OR extra_fee >= 17 OR (name = 'pink' AND extra_fee = 10) )",
    select_only=True,
    dedupe=True,
    schema_tables=['color']
)


In [7]:
# @title Exercise 4
base_exercise_4 = make_df_validator_nospoilers(
    expected_hash='da3f2b5ca4c518fc4804090b55c8050b505f1c776ffb4df361532c8966e44a2b',
    required_cols=['id', 'name', 'parent_id'],
    expected_rows=10,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_4 = base_exercise_4

make_sql_runner(
    conn,
    runner_id="exercise_4",
    description_md="### Exercise 4\n\nWrite a SQL query for table `category` that fulfills the following conditions.  \nAll condition parts must be joined using `AND`.\n\n- Part A: select categories whose `parent_id` is `1` or `6`\n  - Example that should pass this part: `parent_id = 1`\n  - Example that should fail this part: `parent_id = 17`\n\n- Part B: exclude categories named `women`, unless the category id is `7` or `18`\n  - Example that should pass this part: `name = 'women', id = 7`\n  - Example that should fail this part: `name = 'women', id = 9`\n\n- Part C: exclude categories named `trousers`, except when `parent_id` is `17`, but still exclude category `34`\n  - Example that should pass this part: `name = 'trousers', parent_id = 17, id = 33`\n  - Example that should fail this part: `name = 'trousers', id = 34`\n\nReturn: `id`, `name`, `parent_id`\n",
    validator=val_exercise_4,
    sol_sql="SELECT id, name, parent_id FROM category WHERE parent_id IN (1, 6) AND (name <> 'women' OR id IN (7, 18)) AND ( name <> 'trousers' OR (parent_id = 17 AND id <> 34) )",
    select_only=True,
    dedupe=True,
    schema_tables=['category']
)


In [8]:
# @title Exercise 5
base_exercise_5 = make_df_validator_nospoilers(
    expected_hash='33b9cb0c7795f81fadcbe010e3e45dcc3a4badde8010f8a152e6f4e4fda1c663',
    required_cols=['id', 'customer_id', 'clothing_id', 'items', 'order_date'],
    expected_rows=45,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_5 = base_exercise_5

make_sql_runner(
    conn,
    runner_id="exercise_5",
    description_md="description_md: |\n  ### Exercise 5\n\n  Write a SQL query for table `clothing_order` that fulfills the following conditions.  \n  All condition parts must be joined using `AND`.\n\n  - Part A: select orders placed from `2021-05-10` through `2021-06-10`\n    - Example that should pass this part: `order_date = '2021-05-10'`\n    - Example that should fail this part: `order_date = '2021-05-09'`\n\n  - Part B: exclude orders with `5` items, unless the clothing id is `97`\n    - Example that should pass this part: `items = 5, clothing_id = 97`\n    - Example that should fail this part: `items = 5, clothing_id = 50`\n\n  - Part C: exclude orders from customer `48`, except when the order date is in June 2021 and either the clothing id is `23` or the number of items is exactly `2`\n    - Example that should pass this part: `customer_id = 48, order_date = '2021-06-05', clothing_id = 23`\n    - Example that should fail this part: `customer_id = 48, order_date = '2021-06-05', clothing_id = 50, items = 3`\n\n  Return: `id`, `customer_id`, `clothing_id`, `items`, `order_date`\n",
    validator=val_exercise_5,
    sol_sql="SELECT id, customer_id, clothing_id, items, order_date FROM clothing_order WHERE order_date BETWEEN DATE '2021-05-10' AND DATE '2021-06-10' AND (items <> 5 OR clothing_id = 97) AND ( customer_id <> 48 OR ( customer_id = 48 AND order_date >= DATE '2021-06-01' AND order_date <= DATE '2021-06-30' AND (clothing_id = 23 OR items = 2) ) )",
    select_only=True,
    dedupe=True,
    schema_tables=['clothing_order']
)


In [9]:
# @title Exercise 6_1
base_exercise_6_1 = make_df_validator_nospoilers(
    expected_hash='c8f2e45bd94e17ec5c0867936836a7bdac57fe70f0029357712f1f9e88354cf5',
    required_cols=['id', 'name', 'size', 'price', 'color_id', 'category_id'],
    expected_rows=18,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_6_1 = base_exercise_6_1

make_sql_runner(
    conn,
    runner_id="exercise_6_1",
    description_md="description_md: |\n  ### Exercise 6_1\n\n  Write a SQL query for table `clothing` that fulfills the following conditions.  \n  All condition parts must be joined using `AND`.\n\n  - Part A: select clothing items from categories `7`, `8`, `21`, or `22`\n    - Example that should pass this part: `category_id = 7`\n    - Example that should fail this part: `category_id = 10`\n\n  - Part B: exclude black items, unless the name contains `cotton`\n    - Example that should pass this part: `color = 'black', name contains 'cotton'`\n    - Example that should fail this part: `color = 'black', name = 'modern shirt'`\n\n  - Part C: exclude items whose name contains `jacket`, except when the category is `22`, or when the category is `8` and the price is less than `700`\n    - Example that should pass this part: `name = 'wind jacket', category_id = 8, price = 650`\n    - Example that should fail this part: `name = 'winter jacket', category_id = 8, price = 720`\n\n  Return: `id`, `name`, `size`, `price`, `color_id`, `category_id`\n",
    validator=val_exercise_6_1,
    sol_sql="SELECT id, name, size, price, color_id, category_id FROM clothing WHERE category_id IN (7, 8, 21, 22) AND (color_id <> 2 OR name ILIKE '%cotton%') AND ( name NOT ILIKE '%jacket%' OR category_id = 22 OR (category_id = 8 AND price < 700) )",
    select_only=True,
    dedupe=True,
    schema_tables=['clothing']
)


In [10]:
# @title Exercise 6_2
base_exercise_6_2 = make_df_validator_nospoilers(
    expected_hash='66e7bdd478738f5f2e664657a66716a077f9c24b5f8dd21aea83cf8896fb9092',
    required_cols=['id', 'name', 'size', 'price', 'color_id', 'category_id'],
    expected_rows=21,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_6_2 = base_exercise_6_2

make_sql_runner(
    conn,
    runner_id="exercise_6_2",
    description_md="### Exercise 6_2\n\nWrite a SQL query for table `clothing` that fulfills the following conditions.  \nAll condition parts must be joined using `AND`.\n\n- Part A: select clothing items whose category is `7`, `8`, `21`, `22`, `23`, or `24`, and whose price is at least `40`\n  - Example that should pass this part: `category_id = 23, price = 55`\n  - Example that should fail this part: `category_id = 10, price = 55`\n  - Example that should fail this part: `category_id = 23, price = 30`\n\n- Part B: exclude items whose name contains `shirt`, unless the name also contains `cotton` or the color is `white`, but still exclude rows whose name contains both `shirt` and `winter`\n  - Example that should pass this part: `name = 'cotton T-shirt sunrise', color = 'yellow'`\n  - Example that should pass this part: `name = 'shirt blue sky 2', color = 'white'`\n  - Example that should fail this part: `name = 'shirt blue sky 2', color = 'black'`\n  - Example that should fail this part: `name = 'winter shirt', color = 'white'`\n\n- Part C: exclude items with color `pink`, `fuchsia`, or `red`, unless the category is `24`, or the price is greater than `300` and the size is `M` or `L`, but still exclude items from category `24` when the name contains `summer`\n  - Example that should pass this part: `color = 'red', category_id = 24, name = 'layered swim skirt'`\n  - Example that should pass this part: `color = 'pink', price = 455, size = 'M'`\n  - Example that should fail this part: `color = 'fuchsia', price = 200, size = 'M'`\n  - Example that should fail this part: `color = 'red', category_id = 24, name contains 'summer'`\n\nReturn: `id`, `name`, `size`, `price`, `color_id`, `category_id`\n",
    validator=val_exercise_6_2,
    sol_sql="SELECT id, name, size, price, color_id, category_id FROM clothing WHERE category_id IN (7, 8, 21, 22, 23, 24) AND price >= 40 AND ( name NOT ILIKE '%shirt%' OR ( (name ILIKE '%cotton%' OR color_id = 1) AND name NOT ILIKE '%winter%' ) ) AND ( color_id NOT IN (6, 7, 17) OR ( ( category_id = 24 OR (price > 300 AND size IN ('M', 'L')) ) AND NOT (category_id = 24 AND name ILIKE '%summer%') ) )",
    select_only=True,
    dedupe=True,
    schema_tables=['clothing']
)


In [11]:
# @title Exercise 6_3
base_exercise_6_3 = make_df_validator_nospoilers(
    expected_hash='4f45624dddb36edaf0a46b6b6d107306f4dea6a1a291e44b765f4937fdbe56b1',
    required_cols=['id', 'first_name', 'last_name', 'favorite_color_id'],
    expected_rows=20,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_6_3 = base_exercise_6_3

make_sql_runner(
    conn,
    runner_id="exercise_6_3",
    description_md="### Exercise 6_3\n\nWrite a SQL query for table `customer` that fulfills the following conditions.  \nAll condition parts must be joined using `AND`.\n\n- Part A: select customers whose last name is `Johnson`, `Williams`, `Brown`, `Jones`, or `Jackson`, and whose first name has at least `4` letters\n  - Example that should pass this part: `first_name = 'Liam', last_name = 'Johnson'`\n  - Example that should fail this part: `first_name = 'Mia', last_name = 'Brown'`\n  - Example that should fail this part: `first_name = 'Lucas', last_name = 'Taylor'`\n\n- Part B: exclude customers whose favorite color is `teal` or `lavender`, unless the first name starts with `L` or the last name is `Jackson`, but still exclude rows where both the first name starts with `L` and the last name is `Jones`\n  - Example that should pass this part: `favorite color = 'teal', first_name = 'Luna', last_name = 'Smith'`\n  - Example that should pass this part: `favorite color = 'lavender', first_name = 'Ava', last_name = 'Jackson'`\n  - Example that should fail this part: `favorite color = 'teal', first_name = 'Lily', last_name = 'Jones'`\n  - Example that should fail this part: `favorite color = 'lavender', first_name = 'Emma', last_name = 'Brown'`\n\n- Part C: exclude customers whose favorite color is `red`, unless the last name is `Brown` and the first name does not start with `A`, or the first name starts with `E` and the last name is not `Williams`, but still exclude customers whose first name is exactly `Emily`\n  - Example that should pass this part: `favorite color = 'red', first_name = 'Ella', last_name = 'Jones'`\n  - Example that should pass this part: `favorite color = 'red', first_name = 'Liam', last_name = 'Brown'`\n  - Example that should fail this part: `favorite color = 'red', first_name = 'Amelia', last_name = 'Brown'`\n  - Example that should fail this part: `favorite color = 'red', first_name = 'Emily', last_name = 'Brown'`\n\nReturn: `id`, `first_name`, `last_name`, `favorite_color_id`\n",
    validator=val_exercise_6_3,
    sol_sql="SELECT id, first_name, last_name, favorite_color_id FROM customer WHERE last_name IN ('Johnson', 'Williams', 'Brown', 'Jones', 'Jackson') AND LENGTH(first_name) >= 4 AND ( favorite_color_id NOT IN (18, 19) OR ( (first_name ILIKE 'L%' OR last_name = 'Jackson') AND NOT (first_name ILIKE 'L%' AND last_name = 'Jones') ) ) AND ( favorite_color_id <> 17 OR ( ( (last_name = 'Brown' AND first_name NOT ILIKE 'A%') OR (first_name ILIKE 'E%' AND last_name <> 'Williams') ) AND first_name <> 'Emily' ) )",
    select_only=True,
    dedupe=True,
    schema_tables=['customer']
)


# General exercises: conditions, joins, group by, having


In [12]:
# @title Exercise 7
base_exercise_7 = make_df_validator_nospoilers(
    expected_hash='f189b87ee63fdb1d355656de72142ded2b6b40da8b280ba81c6e267ecbf72322',
    required_cols=['clothes', 'color', 'category', 'customers'],
    expected_rows=4,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_7 = base_exercise_7

make_sql_runner(
    conn,
    runner_id="exercise_7",
    description_md='### Exercise 7\n\nWrite a SQL query that selects the following columns:\n\n- the name of the clothing item as `clothes`\n- the clothing color as `color`\n- the clothing category name as `category`\n- the number of different customers who bought that product as `customers`\n\nUse the tables `clothing_order`, `clothing`, `color`, and `category`.\n\nShow only clothing-and-color combinations where the number of customers is greater than `3`.\n\nGroup the result correctly.\n',
    validator=val_exercise_7,
    sol_sql='SELECT g.name AS clothes, r.name AS color, t.name AS category, COUNT(DISTINCT co.customer_id) AS customers FROM clothing_order co JOIN clothing g ON g.id = co.clothing_id JOIN category t ON t.id = g.category_id JOIN color r ON r.id = g.color_id GROUP BY g.name, r.name, t.name HAVING COUNT(DISTINCT co.customer_id) > 3',
    select_only=True,
    dedupe=True,
    schema_tables=['clothing_order', 'clothing', 'color', 'category']
)


In [13]:
# @title Exercise 8
base_exercise_8 = make_df_validator_nospoilers(
    expected_hash='f80b1fa820d95a87cf48f78eb6c298b427fda46207f7b52eaff6fb8ab1590c64',
    required_cols=['name'],
    expected_rows=0,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_8 = base_exercise_8

make_sql_runner(
    conn,
    runner_id="exercise_8",
    description_md='### Exercise 8\n\nWrite a SQL query that selects the name of each main category.\n\nA main category is a category from table `category` where `parent_id` is `NULL`.\n\nUse the tables `category` and `clothing`.\n\nShow only main categories that have more than `2` clothing items assigned to them directly.\n\nReturn:\n- `name`\n',
    validator=val_exercise_8,
    sol_sql='SELECT c.name FROM category c JOIN clothing o ON o.category_id = c.id WHERE c.parent_id IS NULL GROUP BY c.name HAVING COUNT(o.id) > 2',
    select_only=True,
    dedupe=True,
    schema_tables=['category', 'clothing']
)


In [14]:
# @title Exercise 9
base_exercise_9 = make_df_validator_nospoilers(
    expected_hash='70fee27e519904121cc6bb2c04e09578810e4b0a407caef6d9cb13dcc5839ea2',
    required_cols=['last_name', 'first_name', 'total_value'],
    expected_rows=48,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_9 = base_exercise_9

make_sql_runner(
    conn,
    runner_id="exercise_9",
    description_md="### Exercise 9\n\nWrite a SQL query that selects:\n\n- the customer's last name\n- the customer's first name\n- the total value of a single order on items in the customer's favorite color as `total_value`\n\nUse the tables `customer`, `clothing_order`, `clothing`, and `color`.\n\nInclude only:\n- customers where the total value is greater than `1000`\n- or customers who made no purchase, where `total_value` is `NULL`\n\nSort the result by:\n- `last_name`\n- `first_name`\n\nReturn:\n- `last_name`\n- `first_name`\n- `total_value`\n",
    validator=val_exercise_9,
    sol_sql='SELECT cu.last_name, cu.first_name, SUM(s.price + k.extra_fee) * o.items AS total_value FROM customer cu LEFT JOIN clothing_order o ON o.customer_id = cu.id LEFT JOIN clothing s ON s.id = o.clothing_id LEFT JOIN color k ON cu.favorite_color_id = k.id GROUP BY cu.last_name, cu.first_name, o.items HAVING SUM(s.price + k.extra_fee) * o.items > 1000 OR SUM(s.price + k.extra_fee) * o.items IS NULL ORDER BY cu.last_name, cu.first_name',
    select_only=True,
    dedupe=True,
    schema_tables=['customer', 'clothing_order', 'clothing', 'color']
)


In [15]:
# @title Exercise 10
base_exercise_10 = make_df_validator_nospoilers(
    expected_hash='fa515e986c5875f9d2741799b2bd957dff94a399a4b2f37f54fd363547aaa5e6',
    required_cols=['category', 'clothes_count', 'avg_price'],
    expected_rows=6,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_10 = base_exercise_10

make_sql_runner(
    conn,
    runner_id="exercise_10",
    description_md='### Exercise 10\n\nWrite a SQL query that selects:\n\n- the category name as `category`\n- the number of clothing items in that category as `clothes_count`\n- the average clothing price in that category as `avg_price`\n\nUse the tables `category` and `clothing`.\n\nShow only categories that:\n- have at least `3` clothing items\n- and have an average price greater than `200`\n\nSort the result by `avg_price` in descending order.\n\nReturn:\n- `category`\n- `clothes_count`\n- `avg_price`\n',
    validator=val_exercise_10,
    sol_sql='SELECT c.name AS category, COUNT(cl.id) AS clothes_count, AVG(cl.price) AS avg_price FROM category c JOIN clothing cl ON cl.category_id = c.id GROUP BY c.name HAVING COUNT(cl.id) >= 3 AND AVG(cl.price) > 200 ORDER BY avg_price DESC',
    select_only=True,
    dedupe=True,
    schema_tables=['category', 'clothing']
)


In [16]:
# @title Exercise 11
base_exercise_11 = make_df_validator_nospoilers(
    expected_hash='b1bdd890577a6aa9e1696b7063b901f73c7503d99ccfe9d2c81bf1801c45f482',
    required_cols=['color', 'clothes_count', 'max_price'],
    expected_rows=16,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_11 = base_exercise_11

make_sql_runner(
    conn,
    runner_id="exercise_11",
    description_md='### Exercise 11\n\nWrite a SQL query that selects:\n\n- the color name as `color`\n- the number of clothing items in that color as `clothes_count`\n- the highest clothing price in that color as `max_price`\n\nUse the tables `clothing` and `color`.\n\nShow only colors that are used by at least `4` clothing items.\n\nSort the result by:\n- `clothes_count` descending\n- then `color` ascending\n\nReturn:\n- `color`\n- `clothes_count`\n- `max_price`\n',
    validator=val_exercise_11,
    sol_sql='SELECT co.name AS color, COUNT(cl.id) AS clothes_count, MAX(cl.price) AS max_price FROM clothing cl JOIN color co ON co.id = cl.color_id GROUP BY co.name HAVING COUNT(cl.id) >= 4 ORDER BY clothes_count DESC, color ASC',
    select_only=True,
    dedupe=True,
    schema_tables=['clothing', 'color']
)


In [17]:
# @title Exercise 12
base_exercise_12 = make_df_validator_nospoilers(
    expected_hash='e3cc37cd7fa587ac77f9bd992eecb4f929389d26ddde98aace4c2018f0dcd6fd',
    required_cols=['last_name', 'first_name', 'orders_count', 'items_total'],
    expected_rows=24,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_12 = base_exercise_12

make_sql_runner(
    conn,
    runner_id="exercise_12",
    description_md="### Exercise 12\n\nWrite a SQL query that selects:\n\n- the customer's last name\n- the customer's first name\n- the number of orders made by that customer as `orders_count`\n- the total number of items ordered by that customer as `items_total`\n\nUse the tables `customer` and `clothing_order`.\n\nShow only customers who made at least `2` orders and ordered more than `5` items in total.\n\nSort the result by:\n- `items_total` descending\n- then `last_name`\n- then `first_name`\n\nReturn:\n- `last_name`\n- `first_name`\n- `orders_count`\n- `items_total`\n",
    validator=val_exercise_12,
    sol_sql='SELECT c.last_name, c.first_name, COUNT(o.id) AS orders_count, SUM(o.items) AS items_total FROM customer c JOIN clothing_order o ON o.customer_id = c.id GROUP BY c.last_name, c.first_name HAVING COUNT(o.id) >= 2 AND SUM(o.items) > 5 ORDER BY items_total DESC, c.last_name, c.first_name',
    select_only=True,
    dedupe=True,
    schema_tables=['customer', 'clothing_order']
)


In [18]:
# @title Exercise 13
base_exercise_13 = make_df_validator_nospoilers(
    expected_hash='b10df8e1b478c51f90271561ae351293de1b2ae7d96818a72d7a5891cdc4b051',
    required_cols=['category', 'customers'],
    expected_rows=6,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_13 = base_exercise_13

make_sql_runner(
    conn,
    runner_id="exercise_13",
    description_md='### Exercise 13\n\nWrite a SQL query that selects:\n\n- the clothing category name as `category`\n- the number of different customers who bought items from that category as `customers`\n\nUse the tables `clothing_order`, `clothing`, and `category`.\n\nShow only categories bought by more than `4` different customers.\n\nSort the result by `customers` descending.\n\nReturn:\n- `category`\n- `customers`\n',
    validator=val_exercise_13,
    sol_sql='SELECT ca.name AS category, COUNT(DISTINCT co.customer_id) AS customers FROM clothing_order co JOIN clothing cl ON cl.id = co.clothing_id JOIN category ca ON ca.id = cl.category_id GROUP BY ca.name HAVING COUNT(DISTINCT co.customer_id) > 4 ORDER BY customers DESC',
    select_only=True,
    dedupe=True,
    schema_tables=['clothing_order', 'clothing', 'category']
)


In [19]:
# @title Exercise 14
base_exercise_14 = make_df_validator_nospoilers(
    expected_hash='af0e697d92108002be63bbbdf5f912fe717b11933c4abe30aec441f31cb011f8',
    required_cols=['clothes', 'items_sold', 'sales_value'],
    expected_rows=17,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_14 = base_exercise_14

make_sql_runner(
    conn,
    runner_id="exercise_14",
    description_md='### Exercise 14\n\nWrite a SQL query that selects:\n\n- the clothing item name as `clothes`\n- the total number of ordered items as `items_sold`\n- the total sales value as `sales_value`\n\nUse the tables `clothing_order` and `clothing`.\n\nThe total sales value should be calculated as:\n- clothing `price * items`\n\nShow only clothing items where:\n- the total number of ordered items is greater than `5`\n- and the total sales value is greater than `1000`\n\nSort the result by `sales_value` descending.\n\nReturn:\n- `clothes`\n- `items_sold`\n- `sales_value`\n',
    validator=val_exercise_14,
    sol_sql='SELECT c.name AS clothes, SUM(o.items) AS items_sold, SUM(c.price * o.items) AS sales_value FROM clothing_order o JOIN clothing c ON c.id = o.clothing_id GROUP BY c.name HAVING SUM(o.items) > 5 AND SUM(c.price * o.items) > 1000 ORDER BY sales_value DESC',
    select_only=True,
    dedupe=True,
    schema_tables=['clothing_order', 'clothing']
)


In [20]:
# @title Exercise 15
base_exercise_15 = make_df_validator_nospoilers(
    expected_hash='58c45260c1105cb50ebc4921a47be12aedb64cfb70e35af784a6ff17ae3f4264',
    required_cols=['favorite_color', 'customers_count'],
    expected_rows=12,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_15 = base_exercise_15

make_sql_runner(
    conn,
    runner_id="exercise_15",
    description_md="### Exercise 15\n\nWrite a SQL query that selects:\n\n- the customer's favorite color name as `favorite_color`\n- the number of customers with that favorite color as `customers_count`\n\nUse the tables `customer` and `color`.\n\nShow only favorite colors chosen by at least `3` customers.\n\nSort the result by:\n- `customers_count` descending\n- then `favorite_color` ascending\n\nReturn:\n- `favorite_color`\n- `customers_count`\n",
    validator=val_exercise_15,
    sol_sql='SELECT co.name AS favorite_color, COUNT(cu.id) AS customers_count FROM customer cu JOIN color co ON co.id = cu.favorite_color_id GROUP BY co.name HAVING COUNT(cu.id) >= 3 ORDER BY customers_count DESC, favorite_color ASC',
    select_only=True,
    dedupe=True,
    schema_tables=['customer', 'color']
)


In [21]:
# @title Exercise 16
base_exercise_16 = make_df_validator_nospoilers(
    expected_hash='de00b29218ac087722cae01176ae20eae5f30381bfbf4ab12216f7b3a3689306',
    required_cols=['category', 'color', 'clothes_count'],
    expected_rows=28,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_16 = base_exercise_16

make_sql_runner(
    conn,
    runner_id="exercise_16",
    description_md='### Exercise 16\n\nWrite a SQL query that selects:\n\n- the category name as `category`\n- the color name as `color`\n- the number of clothing items for that category-and-color combination as `clothes_count`\n\nUse the tables `clothing`, `category`, and `color`.\n\nShow only category-and-color combinations that contain at least `2` clothing items.\n\nSort the result by:\n- `category`\n- then `color`\n\nReturn:\n- `category`\n- `color`\n- `clothes_count`\n',
    validator=val_exercise_16,
    sol_sql='SELECT ca.name AS category, co.name AS color, COUNT(cl.id) AS clothes_count FROM clothing cl JOIN category ca ON ca.id = cl.category_id JOIN color co ON co.id = cl.color_id GROUP BY ca.name, co.name HAVING COUNT(cl.id) >= 2 ORDER BY category, color',
    select_only=True,
    dedupe=True,
    schema_tables=['clothing', 'category', 'color']
)


# Self-joins


In [22]:
# @title Exercise 17
base_exercise_17 = make_df_validator_nospoilers(
    expected_hash='f5faf449251da6c8c966cca74ae7dc6630b3bb10597d09cb9b40a133fd4e2f56',
    required_cols=['subcategory', 'parent_category'],
    expected_rows=38,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_17 = base_exercise_17

make_sql_runner(
    conn,
    runner_id="exercise_17",
    description_md='### Exercise 17\n\nThe `category` table stores hierarchical data using the `parent_id` column.\n\nWrite a SQL query that returns:\n\n- the name of each subcategory as `subcategory`\n- the name of its parent category as `parent_category`\n\nA subcategory is any category where `parent_id` is not NULL.\n\nUse a self join on the `category` table to match each subcategory with its parent.\n\nSort the result by:\n- `parent_category`\n- then `subcategory`\n',
    validator=val_exercise_17,
    sol_sql='SELECT c.name AS subcategory, p.name AS parent_category FROM category c JOIN category p ON c.parent_id = p.id ORDER BY parent_category, subcategory',
    select_only=True,
    dedupe=True,
    schema_tables=['category']
)


In [23]:
# @title Exercise 18
base_exercise_18 = make_df_validator_nospoilers(
    expected_hash='379abe68c3fb5b9c56162d2cadfeba2ee0d5d9570c2edcf44300b294564493a5',
    required_cols=['subcategory', 'main_category'],
    expected_rows=13,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_18 = base_exercise_18

make_sql_runner(
    conn,
    runner_id="exercise_18",
    description_md='### Exercise 18\n\nThe `category` table includes both main categories and subcategories.\n\nWrite a SQL query that returns:\n\n- the name of each subcategory as `subcategory`\n- the name of the main category it belongs to as `main_category`\n\nA main category is a category where `parent_id` is NULL.\n\nUse a self join to connect each subcategory to its parent, and filter so that only parents that are main categories are included.\n\nSort the result by:\n- `main_category`\n- then `subcategory`\n',
    validator=val_exercise_18,
    sol_sql='SELECT c.name AS subcategory, p.name AS main_category FROM category c JOIN category p ON c.parent_id = p.id WHERE p.parent_id IS NULL ORDER BY main_category, subcategory',
    select_only=True,
    dedupe=True,
    schema_tables=['category']
)


In [24]:
# @title Exercise 19
base_exercise_19 = make_df_validator_nospoilers(
    expected_hash='b8fc370c418828bba0b89a50eb79e2af071ac9325f1b2ac2f2c423d927e07891',
    required_cols=['category', 'parent_category'],
    expected_rows=31,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_19 = base_exercise_19

make_sql_runner(
    conn,
    runner_id="exercise_19",
    description_md='### Exercise 19\n\nEach clothing item belongs to a category, and categories can have parent categories.\n\nWrite a SQL query that returns:\n\n- the category name as `category`\n- the parent category name as `parent_category`\n\nOnly include categories that are actually used by at least one clothing item.\n\nUse the `clothing` table and a self join on the `category` table to link each category to its parent.\n\nRemove duplicates from the result.\n\nSort the result by:\n- `parent_category`\n- then `category`\n',
    validator=val_exercise_19,
    sol_sql='SELECT DISTINCT c.name AS category, p.name AS parent_category FROM clothing cl JOIN category c ON c.id = cl.category_id JOIN category p ON c.parent_id = p.id ORDER BY parent_category, category',
    select_only=True,
    dedupe=True,
    schema_tables=['clothing', 'category']
)


In [25]:
# @title Exercise 20
base_exercise_20 = make_df_validator_nospoilers(
    expected_hash='77a42acfe5b7bda56d0b05cba2fae2d6d88e2afffc03cd7475f5eb5becdf28da',
    required_cols=['parent_category', 'subcategories_count'],
    expected_rows=11,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_20 = base_exercise_20

make_sql_runner(
    conn,
    runner_id="exercise_20",
    description_md='### Exercise 20\n\nCategories can have multiple subcategories.\n\nWrite a SQL query that returns:\n\n- the name of each parent category as `parent_category`\n- the number of subcategories it has as `subcategories_count`\n\nUse a self join on the `category` table to match each parent with its subcategories.\n\nOnly include parent categories that have at least 2 subcategories.\n\nSort the result by:\n- `subcategories_count` descending\n',
    validator=val_exercise_20,
    sol_sql='SELECT p.name AS parent_category, COUNT(c.id) AS subcategories_count FROM category c JOIN category p ON c.parent_id = p.id GROUP BY p.name HAVING COUNT(c.id) >= 2 ORDER BY subcategories_count DESC',
    select_only=True,
    dedupe=True,
    schema_tables=['category']
)


In [26]:
# @title Exercise 21
base_exercise_21 = make_df_validator_nospoilers(
    expected_hash='2a1ad50e24dc803534dadcc266d7e30904286a9b78a132a625c03f8b88fd4b0a',
    required_cols=['category', 'parent_category', 'clothes_count'],
    expected_rows=26,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_21 = base_exercise_21

make_sql_runner(
    conn,
    runner_id="exercise_21",
    description_md='### Exercise 21\n\nEach clothing item belongs to a category, and categories can have parent categories.\n\nWrite a SQL query that returns:\n\n- the category name as `category`\n- the parent category name as `parent_category`\n- the number of clothing items in that category as `clothes_count`\n\nUse a self join on the `category` table to retrieve the parent category.\n\nOnly include categories that:\n- have a parent category\n- and contain at least 2 clothing items\n\nSort the result by:\n- `clothes_count` descending\n',
    validator=val_exercise_21,
    sol_sql='SELECT c.name AS category, p.name AS parent_category, COUNT(cl.id) AS clothes_count FROM clothing cl JOIN category c ON c.id = cl.category_id JOIN category p ON c.parent_id = p.id GROUP BY c.name, p.name HAVING COUNT(cl.id) >= 2 ORDER BY clothes_count DESC',
    select_only=True,
    dedupe=True,
    schema_tables=['clothing', 'category']
)


In [27]:
# @title Exercise 22
base_exercise_22 = make_df_validator_nospoilers(
    expected_hash='dcfa18f0bb16d1a72fc7668d15c973d54253766989ecbdae9a4f2190026603da',
    required_cols=['category', 'parent_category', 'main_category'],
    expected_rows=25,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_22 = base_exercise_22

make_sql_runner(
    conn,
    runner_id="exercise_22",
    description_md='### Exercise 22\n\nThe category hierarchy can have multiple levels:\ncategory → parent → main category.\n\nWrite a SQL query that returns:\n\n- the category name as `category`\n- the parent category name as `parent_category`\n- the top-level (main) category name as `main_category`\n\nUse the `category` table and self joins to:\n- match each category to its parent\n- and the parent to its own parent\n\nOnly include categories that have both a parent and a main category.\n\nSort the result by:\n- `main_category`\n- then `parent_category`\n- then `category`\n',
    validator=val_exercise_22,
    sol_sql='SELECT c.name AS category, p.name AS parent_category, m.name AS main_category FROM category c JOIN category p ON c.parent_id = p.id JOIN category m ON p.parent_id = m.id ORDER BY main_category, parent_category, category',
    select_only=True,
    dedupe=True,
    schema_tables=['category']
)


# Subqueries


In [28]:
# @title Exercise 23
base_exercise_23 = make_df_validator_nospoilers(
    expected_hash='f3b8fd4483a3a95acc25260e7a88adc60c4463f8801ef45d20de69b4e67cc37f',
    required_cols=['id', 'name', 'price', 'category_id'],
    expected_rows=16,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_23 = base_exercise_23

make_sql_runner(
    conn,
    runner_id="exercise_23",
    description_md='### Exercise 23\n\nWrite a SQL query for table `clothing`.\n\nReturn:\n- `id`\n- `name`\n- `price`\n- `category_id`\n\nSelect clothing items that belong to a category whose name is `women`.\n\nYou should not type category IDs by hand.  \nThe category IDs must be retrieved from table `category` based on its condition.\n\nUse a subquery in `WHERE`.\n',
    validator=val_exercise_23,
    sol_sql="SELECT id, name, price, category_id FROM clothing WHERE category_id IN ( SELECT id FROM category WHERE name = 'women' )",
    select_only=True,
    dedupe=True,
    schema_tables=['clothing', 'category']
)


In [29]:
# @title Exercise 24
base_exercise_24 = make_df_validator_nospoilers(
    expected_hash='47226c05e087f5ee17e05291f2bbce3a330fb8b0cf17db2ed0321e610a83c4b5',
    required_cols=['id', 'name', 'size', 'price', 'color_id'],
    expected_rows=73,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_24 = base_exercise_24

make_sql_runner(
    conn,
    runner_id="exercise_24",
    description_md='### Exercise 24\n\nWrite a SQL query for table `clothing`.\n\nReturn:\n- `id`\n- `name`\n- `size`\n- `price`\n- `color_id`\n\nSelect clothing items whose color is used by customers as a favorite color by at least `4` customers.\n\nYou should not type color IDs by hand.  \nThe valid color IDs must be computed from table `customer`.\n\nUse a subquery in `WHERE`.\n',
    validator=val_exercise_24,
    sol_sql='SELECT id, name, size, price, color_id FROM clothing WHERE color_id IN ( SELECT favorite_color_id FROM customer GROUP BY favorite_color_id HAVING COUNT(*) >= 4 )',
    select_only=True,
    dedupe=True,
    schema_tables=['clothing', 'customer']
)


In [30]:
# @title Exercise 25
base_exercise_25 = make_df_validator_nospoilers(
    expected_hash='667eab956b1d8e9b40a094419cf75b5bc05e37de6fdf98ec8742e254753b54cc',
    required_cols=['id', 'name', 'price'],
    expected_rows=50,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_exercise_25 = base_exercise_25

make_sql_runner(
    conn,
    runner_id="exercise_25",
    description_md="### Exercise 25\n\nWrite a SQL query for table `clothing`.\n\nReturn:\n- `id`\n- `name`\n- `price`\n\nSelect clothing items whose price is greater than the average price of all clothing items in the same category.\n\nThe comparison value is not stored in the table.  \nIt must be computed for each row based on that row's category.\n\nUse a subquery in `WHERE`.\n",
    validator=val_exercise_25,
    sol_sql='SELECT c1.id, c1.name, c1.price FROM clothing c1 WHERE c1.price > ( SELECT AVG(c2.price) FROM clothing c2 WHERE c2.category_id = c1.category_id )',
    select_only=True,
    dedupe=True,
    schema_tables=['clothing']
)
